**Streaming Platform Analytics**

In [1]:
# Setup 1 - Install PySpark
!pip install pyspark

In [2]:
# Setup 2 - Imports and Spark session
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel

spark = SparkSession.builder.appName("Assessment2").getOrCreate()

In [3]:
# Setup 3 - Upload the six CSV files
uploaded = files.upload()

Saving content_catalog_120.csv to content_catalog_120.csv
Saving device_master_8.csv to device_master_8.csv
Saving subscriber_master_2400.csv to subscriber_master_2400.csv
Saving subscription_plans_4.csv to subscription_plans_4.csv
Saving watch_sessions_80000.csv to watch_sessions_80000.csv
Saving watch_sessions_extra_10000.csv to watch_sessions_extra_10000.csv


In [4]:
# Task 01 - Load all six CSVs as Spark DataFrames (and preview main sessions)
def read_csv(name):
    return spark.read.csv(name, header=True, inferSchema=True)

sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")

print("Main sessions:", sessions_df.count())
sessions_df.printSchema()
sessions_df.show()

Main sessions: 80000
root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------

In [5]:
# Task 01 - Schema, row count and first 10 records of sessions_df
sessions_df.printSchema()

print("Row count:", sessions_df.count())

sessions_df.show(10)

root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

Row count: 80000
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+---

In [6]:
# Task 01 - Identify numeric, string and timestamp fields
numeric_cols = [name for name, dtype in sessions_df.dtypes if dtype in ("int", "bigint", "double", "float")]
string_cols = [name for name, dtype in sessions_df.dtypes if dtype == "string"]
timestamp_cols = [name for name, dtype in sessions_df.dtypes if dtype in ("timestamp", "date")]

print("Numeric:", numeric_cols)
print("String:", string_cols)
print("Timestamp:", timestamp_cols)

Numeric: ['watch_minutes', 'completion_pct', 'buffering_seconds', 'data_used_mb', 'user_rating']
String: ['session_id', 'subscriber_id', 'content_id', 'device_id', 'stream_quality', 'session_status']
Timestamp: ['session_start']


In [7]:
# Task 02 - Select columns; filter Completed sessions with watch_minutes > 60
completed_long_df = sessions_df \
    .select("session_id", "subscriber_id", "content_id", "watch_minutes", "session_status") \
    .filter((F.col("session_status") == "Completed") & (F.col("watch_minutes") > 60))

completed_long_df.show()

completed_long_df.count()

+----------+-------------+----------+-------------+--------------+
|session_id|subscriber_id|content_id|watch_minutes|session_status|
+----------+-------------+----------+-------------+--------------+
|WS00000049|     SUB01859|    CT0075|          125|     Completed|
|WS00000179|     SUB00935|    CT0054|           62|     Completed|
|WS00000184|     SUB00072|    CT0119|           91|     Completed|
|WS00000222|     SUB01603|    CT0103|          116|     Completed|
|WS00000233|     SUB01233|    CT0027|          158|     Completed|
|WS00000244|     SUB00985|    CT0048|          140|     Completed|
|WS00000249|     SUB00681|    CT0100|          120|     Completed|
|WS00000257|     SUB02115|    CT0120|           68|     Completed|
|WS00000292|     SUB01068|    CT0048|          141|     Completed|
|WS00000326|     SUB00368|    CT0096|           97|     Completed|
|WS00000415|     SUB01758|    CT0105|          150|     Completed|
|WS00000419|     SUB00367|    CT0036|           99|     Comple

3403

In [8]:
# Task 03 - watch_hours and high_engagement flag (nulls in completion_pct preserved)
engagement_df = sessions_df \
    .withColumn("watch_hours", F.round(F.col("watch_minutes") / 60, 2)) \
    .withColumn(
        "high_engagement",
        F.when(F.col("completion_pct") >= 75, 1)
         .when(F.col("completion_pct") < 75, 0)
    )

engagement_df.select("session_id", "watch_minutes", "watch_hours", "completion_pct", "high_engagement").show(10)

engagement_df.groupBy("high_engagement").count().orderBy("high_engagement").show()

+----------+-------------+-----------+--------------+---------------+
|session_id|watch_minutes|watch_hours|completion_pct|high_engagement|
+----------+-------------+-----------+--------------+---------------+
|WS00000001|            6|        0.1|          11.8|              0|
|WS00000002|           31|       0.52|          56.4|              0|
|WS00000003|           29|       0.48|          63.0|              0|
|WS00000004|           69|       1.15|          75.0|              1|
|WS00000005|           36|        0.6|          23.4|              0|
|WS00000006|           20|       0.33|          19.8|              0|
|WS00000007|           78|        1.3|          83.9|              1|
|WS00000008|           22|       0.37|          66.7|              0|
|WS00000009|           21|       0.35|          80.8|              1|
|WS00000010|           21|       0.35|          65.6|              0|
+----------+-------------+-----------+--------------+---------------+
only showing top 10 

In [9]:
# Task 04 - Sort by data_used_mb descending; top 20 records
sessions_df.orderBy(F.col("data_used_mb").desc()).show(20)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00051169|     SUB00155|    CT0069|     DV08|2026-07-20 00:11:00|          167|          97.7|              9.0|            4K|     Completed|      8881.8|        3.0|
|WS00074513|     SUB00684|    CT0056|     DV05|2026-08-09 21:53:00|          157|          96.9|              8.0|            4K|     Completed|      8408.4|       NULL|
|WS00039201|     SUB01664|    CT0027|     DV08|2026-05-16 06:35:00|          161|          97.0|              9.0|            4K|     Completed|      

In [10]:
# Task 05 - Missing values in subscriber_id, device_id, completion_pct, buffering_seconds, user_rating
sessions_df.select(
    F.count(F.when(F.col("subscriber_id").isNull(), 1)).alias("missing_subscriber_id"),
    F.count(F.when(F.col("device_id").isNull(), 1)).alias("missing_device_id"),
    F.count(F.when(F.col("completion_pct").isNull(), 1)).alias("missing_completion_pct"),
    F.count(F.when(F.col("buffering_seconds").isNull(), 1)).alias("missing_buffering_seconds"),
    F.count(F.when(F.col("user_rating").isNull(), 1)).alias("missing_user_rating")
).show()

+---------------------+-----------------+----------------------+-------------------------+-------------------+
|missing_subscriber_id|missing_device_id|missing_completion_pct|missing_buffering_seconds|missing_user_rating|
+---------------------+-----------------+----------------------+-------------------------+-------------------+
|                  951|             2003|                  3513|                     4424|              77605|
+---------------------+-----------------+----------------------+-------------------------+-------------------+



In [11]:
# Task 06 - New DataFrame with missing buffering_seconds replaced by 0 (original untouched)
buffering_filled_df = sessions_df.na.fill({"buffering_seconds": 0})

print("Nulls in new DataFrame:", buffering_filled_df.filter(F.col("buffering_seconds").isNull()).count())
print("Nulls in original:", sessions_df.filter(F.col("buffering_seconds").isNull()).count())

Nulls in new DataFrame: 0
Nulls in original: 4424


In [12]:
# Task 07 - Exclude sessions with missing device_id; compare record counts
device_known_df = sessions_df.na.drop(subset=["device_id"])

print("Before:", sessions_df.count())
print("After:", device_known_df.count())
print("Removed:", sessions_df.count() - device_known_df.count())

Before: 80000
After: 77997
Removed: 2003


In [13]:
# Task 08 - completion_label using when() / otherwise()
completion_label_df = sessions_df.withColumn(
    "completion_label",
    F.when(F.col("completion_pct").isNull(), "Unknown")
     .when(F.col("completion_pct") < 30, "Low")
     .when(F.col("completion_pct") < 90, "Medium")
     .otherwise("High")
)

completion_label_df.groupBy("completion_label").count().orderBy(F.col("count").desc()).show()

+----------------+-----+
|completion_label|count|
+----------------+-----+
|          Medium|57103|
|             Low|13267|
|            High| 6117|
|         Unknown| 3513|
+----------------+-----+



In [14]:
# Task 09 - Join with content catalog: session count and total watch_minutes by genre
sessions_df.join(content_df, on="content_id", how="inner") \
    .groupBy("genre") \
    .agg(
        F.count("*").alias("session_count"),
        F.sum("watch_minutes").alias("total_watch_minutes")
    ) \
    .orderBy(F.col("total_watch_minutes").desc()) \
    .show()

+-----------+-------------+-------------------+
|      genre|session_count|total_watch_minutes|
+-----------+-------------+-------------------+
|  Adventure|        15561|             839700|
|      Drama|        10733|             519872|
|  Animation|        11886|             490335|
|     Comedy|         9105|             452563|
|     Sci-Fi|         9344|             433946|
|     Sports|         9967|             383044|
|Documentary|         7432|             256626|
|   Thriller|         5972|             248430|
+-----------+-------------+-------------------+



In [15]:
# Task 10 - Join with device master: avg buffering_seconds and avg watch_minutes by device_type
sessions_df.join(devices_df, on="device_id", how="inner") \
    .groupBy("device_type") \
    .agg(
        F.round(F.avg("buffering_seconds"), 2).alias("avg_buffering_seconds"),
        F.round(F.avg("watch_minutes"), 2).alias("avg_watch_minutes")
    ) \
    .orderBy("device_type") \
    .show()

+---------------+---------------------+-----------------+
|    device_type|avg_buffering_seconds|avg_watch_minutes|
+---------------+---------------------+-----------------+
|  Android Phone|                35.68|            45.09|
|Desktop Browser|                36.03|            45.57|
|   Game Console|                35.51|            45.04|
| Laptop Browser|                36.93|            45.84|
|       Smart TV|                36.67|            45.34|
|Streaming Stick|                36.32|            45.09|
|         Tablet|                35.24|            45.39|
|         iPhone|                35.76|            45.23|
+---------------+---------------------+-----------------+



In [16]:
# Task 11 - Join sessions, subscribers, plans: completion rate by plan_name (non-null subscriber_id)
sessions_df.filter(F.col("subscriber_id").isNotNull()) \
    .join(subscribers_df, on="subscriber_id", how="inner") \
    .join(plans_df, on="plan_id", how="inner") \
    .groupBy("plan_name") \
    .agg(
        F.count("*").alias("total_sessions"),
        F.sum(F.when(F.col("session_status") == "Completed", 1).otherwise(0)).alias("completed_sessions")
    ) \
    .withColumn(
        "completion_rate_pct",
        F.round(F.col("completed_sessions") / F.col("total_sessions") * 100, 2)
    ) \
    .orderBy("plan_name") \
    .show()

+---------+--------------+------------------+-------------------+
|plan_name|total_sessions|completed_sessions|completion_rate_pct|
+---------+--------------+------------------+-------------------+
|    Basic|         25086|              2006|                8.0|
|   Mobile|         17510|              1429|               8.16|
|  Premium|         12198|               975|               7.99|
| Standard|         23684|              1868|               7.89|
+---------+--------------+------------------+-------------------+



In [17]:
# Task 12 - Monthly recurring revenue by plan for Active subscribers (each counted once)
mrr_df = subscribers_df.filter(F.col("subscriber_status") == "Active") \
    .join(plans_df, on="plan_id", how="inner") \
    .groupBy("plan_id", "plan_name", "monthly_fee_inr") \
    .agg(
        F.count("*").alias("active_subscribers"),
        F.sum("monthly_fee_inr").alias("monthly_recurring_revenue_inr")
    ) \
    .orderBy("plan_id")

mrr_df.show()

mrr_df.agg(F.sum("monthly_recurring_revenue_inr").alias("total_mrr_inr")).show()

+-------+---------+---------------+------------------+-----------------------------+
|plan_id|plan_name|monthly_fee_inr|active_subscribers|monthly_recurring_revenue_inr|
+-------+---------+---------------+------------------+-----------------------------+
|   PL01|   Mobile|            149|               443|                        66007|
|   PL02|    Basic|            249|               632|                       157368|
|   PL03| Standard|            399|               604|                       240996|
|   PL04|  Premium|            649|               326|                       211574|
+-------+---------+---------------+------------------+-----------------------------+

+-------------+
|total_mrr_inr|
+-------------+
|       675945|
+-------------+



In [18]:
# Task 13 - Unmatched subscriber_id (sessions with no subscriber master record)
unmatched_subscribers_df = sessions_df \
    .filter(F.col("subscriber_id").isNotNull()) \
    .join(subscribers_df, on="subscriber_id", how="left_anti")

print("Sessions with an unmatched subscriber_id:", unmatched_subscribers_df.count())

unmatched_subscribers_df.groupBy("subscriber_id").count().show()

Sessions with an unmatched subscriber_id: 571
+-------------+-----+
|subscriber_id|count|
+-------------+-----+
|     SUB99999|  571|
+-------------+-----+



In [19]:
# Task 13 - Unmatched device_id (sessions with no device master record)
unmatched_devices_df = sessions_df \
    .filter(F.col("device_id").isNotNull()) \
    .join(devices_df, on="device_id", how="left_anti")

print("Sessions with an unmatched device_id:", unmatched_devices_df.count())

unmatched_devices_df.groupBy("device_id").count().show()

Sessions with an unmatched device_id: 481
+---------+-----+
|device_id|count|
+---------+-----+
|     DV99|  481|
+---------+-----+



In [20]:
# Imports needed for Part D and E (time, os)
import time
import os

In [21]:
# Task 14 - Convert session_start to timestamp; extract year and month
timestamp_df = sessions_df \
    .withColumn("session_start", F.to_timestamp("session_start", "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("session_year", F.year("session_start")) \
    .withColumn("session_month", F.month("session_start"))

timestamp_df.select("session_id", "session_start", "session_year", "session_month").show()

+----------+-------------------+------------+-------------+
|session_id|      session_start|session_year|session_month|
+----------+-------------------+------------+-------------+
|WS00000001|2026-04-26 18:59:00|        2026|            4|
|WS00000002|2026-07-30 06:26:00|        2026|            7|
|WS00000003|2026-02-22 20:55:00|        2026|            2|
|WS00000004|2026-02-20 17:07:00|        2026|            2|
|WS00000005|2026-04-06 04:33:00|        2026|            4|
|WS00000006|2026-03-17 07:54:00|        2026|            3|
|WS00000007|2026-02-25 04:38:00|        2026|            2|
|WS00000008|2026-08-22 00:06:00|        2026|            8|
|WS00000009|2026-06-13 23:28:00|        2026|            6|
|WS00000010|2026-08-09 09:54:00|        2026|            8|
|WS00000011|2026-03-05 17:13:00|        2026|            3|
|WS00000012|2026-06-16 16:49:00|        2026|            6|
|WS00000013|2026-02-05 15:05:00|        2026|            2|
|WS00000014|2026-09-25 23:34:00|        

In [22]:
# Task 14 - Monthly session counts and total watch_minutes for 2026
timestamp_df.filter(F.col("session_year") == 2026) \
    .groupBy("session_year", "session_month") \
    .agg(
        F.count("*").alias("session_count"),
        F.sum("watch_minutes").alias("total_watch_minutes")
    ) \
    .orderBy("session_month") \
    .show()

+------------+-------------+-------------+-------------------+
|session_year|session_month|session_count|total_watch_minutes|
+------------+-------------+-------------+-------------------+
|        2026|            1|         9070|             407215|
|        2026|            2|         8218|             372726|
|        2026|            3|         9019|             403971|
|        2026|            4|         8817|             399432|
|        2026|            5|         9006|             407386|
|        2026|            6|         8776|             404203|
|        2026|            7|         9204|             416104|
|        2026|            8|         8996|             407374|
|        2026|            9|         8894|             406105|
+------------+-------------+-------------+-------------------+



In [23]:
# Task 15 - Join signup_date; days between signup and each session (earliest sessions)
signup_df = timestamp_df \
    .join(subscribers_df, on="subscriber_id", how="inner") \
    .withColumn(
        "days_since_signup",
        F.datediff(F.to_date("session_start"), F.to_date("signup_date"))
    )

signup_df.select("session_id", "subscriber_id", "session_start", "signup_date", "days_since_signup") \
    .orderBy("session_start", "session_id") \
    .show()

+----------+-------------+-------------------+-----------+-----------------+
|session_id|subscriber_id|      session_start|signup_date|days_since_signup|
+----------+-------------+-------------------+-----------+-----------------+
|WS00035235|     SUB01707|2026-01-01 00:00:00| 2024-02-13|              688|
|WS00032058|     SUB01248|2026-01-01 00:04:00| 2024-03-17|              655|
|WS00074739|     SUB02206|2026-01-01 00:04:00| 2025-08-31|              123|
|WS00048560|     SUB02342|2026-01-01 00:14:00| 2025-08-25|              129|
|WS00067087|     SUB00563|2026-01-01 00:15:00| 2024-08-11|              508|
|WS00004245|     SUB00711|2026-01-01 00:18:00| 2024-10-26|              432|
|WS00041710|     SUB00261|2026-01-01 00:21:00| 2025-05-15|              231|
|WS00045132|     SUB02167|2026-01-01 00:28:00| 2025-11-22|               40|
|WS00036438|     SUB01237|2026-01-01 00:41:00| 2024-02-01|              700|
|WS00030050|     SUB00646|2026-01-01 00:51:00| 2024-11-20|              407|

In [24]:
# Task 15 - (repeat of previous cell - identical code)
signup_df = timestamp_df \
    .join(subscribers_df, on="subscriber_id", how="inner") \
    .withColumn(
        "days_since_signup",
        F.datediff(F.to_date("session_start"), F.to_date("signup_date"))
    )

signup_df.select("session_id", "subscriber_id", "session_start", "signup_date", "days_since_signup") \
    .orderBy("session_start", "session_id") \
    .show()

+----------+-------------+-------------------+-----------+-----------------+
|session_id|subscriber_id|      session_start|signup_date|days_since_signup|
+----------+-------------+-------------------+-----------+-----------------+
|WS00035235|     SUB01707|2026-01-01 00:00:00| 2024-02-13|              688|
|WS00032058|     SUB01248|2026-01-01 00:04:00| 2024-03-17|              655|
|WS00074739|     SUB02206|2026-01-01 00:04:00| 2025-08-31|              123|
|WS00048560|     SUB02342|2026-01-01 00:14:00| 2025-08-25|              129|
|WS00067087|     SUB00563|2026-01-01 00:15:00| 2024-08-11|              508|
|WS00004245|     SUB00711|2026-01-01 00:18:00| 2024-10-26|              432|
|WS00041710|     SUB00261|2026-01-01 00:21:00| 2025-05-15|              231|
|WS00045132|     SUB02167|2026-01-01 00:28:00| 2025-11-22|               40|
|WS00036438|     SUB01237|2026-01-01 00:41:00| 2024-02-01|              700|
|WS00030050|     SUB00646|2026-01-01 00:51:00| 2024-11-20|              407|

In [25]:
# Task 16 - Weekday vs Weekend classification; compare average watch_minutes
daytype_df = timestamp_df.withColumn(
    "day_type",
    F.when(F.dayofweek("session_start").isin(1, 7), "Weekend").otherwise("Weekday")
)

daytype_df.groupBy("day_type") \
    .agg(
        F.count("*").alias("session_count"),
        F.round(F.avg("watch_minutes"), 2).alias("avg_watch_minutes")
    ) \
    .orderBy("day_type") \
    .show()

+--------+-------------+-----------------+
|day_type|session_count|avg_watch_minutes|
+--------+-------------+-----------------+
| Weekday|        57228|            45.26|
| Weekend|        22772|            45.41|
+--------+-------------+-----------------+



In [26]:
# Task 17 - Total watch_minutes by genre and content_title; top 3 per genre (dense_rank)
title_totals_df = sessions_df \
    .join(content_df, on="content_id", how="inner") \
    .groupBy("genre", "content_title") \
    .agg(F.sum("watch_minutes").alias("total_watch_minutes"))

title_window = Window.partitionBy("genre").orderBy(F.col("total_watch_minutes").desc())

title_totals_df \
    .withColumn("title_rank", F.dense_rank().over(title_window)) \
    .filter(F.col("title_rank") <= 3) \
    .orderBy("genre", "title_rank") \
    .show()

+-----------+----------------+-------------------+----------+
|      genre|   content_title|total_watch_minutes|title_rank|
+-----------+----------------+-------------------+----------+
|  Adventure|Infinite Horizon|              66395|         1|
|  Adventure|    Lost Journey|              61823|         2|
|  Adventure|    Solar Memory|              60540|         3|
|  Animation|      Last River|              51398|         1|
|  Animation| Emerald Circuit|              51310|         2|
|  Animation|   Golden Puzzle|              50515|         3|
|     Comedy|    Blue Horizon|              61428|         1|
|     Comedy|   Bright Valley|              60921|         2|
|     Comedy|   Silver Valley|              60422|         3|
|Documentary|   Fading Signal|              47085|         1|
|Documentary|  Emerald Valley|              40819|         2|
|Documentary|   Solar Horizon|              36152|         3|
|      Drama|  Crimson Summit|              65724|         1|
|      D

In [27]:
# Task 18 - Most recent session per non-null subscriber_id (row_number)
latest_window = Window.partitionBy("subscriber_id") \
    .orderBy(F.col("session_start").desc(), F.col("session_id").desc())

latest_sessions_df = timestamp_df \
    .filter(F.col("subscriber_id").isNotNull()) \
    .withColumn("row_num", F.row_number().over(latest_window)) \
    .filter(F.col("row_num") == 1)

latest_sessions_df.select("subscriber_id", "session_id", "session_start", "watch_minutes") \
    .orderBy("subscriber_id") \
    .show()

latest_sessions_df.count()

+-------------+----------+-------------------+-------------+
|subscriber_id|session_id|      session_start|watch_minutes|
+-------------+----------+-------------------+-------------+
|     SUB00001|WS00020822|2026-09-11 19:19:00|           34|
|     SUB00002|WS00032744|2026-09-02 05:41:00|           45|
|     SUB00003|WS00006110|2026-09-30 03:13:00|           11|
|     SUB00004|WS00037660|2026-09-30 17:48:00|           62|
|     SUB00005|WS00070939|2026-09-18 02:50:00|           22|
|     SUB00006|WS00057918|2026-09-23 06:26:00|          104|
|     SUB00007|WS00024658|2026-09-23 20:45:00|           55|
|     SUB00008|WS00032330|2026-09-26 22:41:00|           41|
|     SUB00009|WS00008916|2026-09-04 22:21:00|           30|
|     SUB00010|WS00049685|2026-09-29 06:36:00|            6|
|     SUB00011|WS00052168|2026-09-28 11:38:00|           51|
|     SUB00012|WS00022826|2026-09-09 05:02:00|           69|
|     SUB00013|WS00045767|2026-09-25 12:55:00|           17|
|     SUB00014|WS0005950

2401

In [28]:
# Task 19 - Compare watch_minutes with previous session per subscriber (lag)
lag_window = Window.partitionBy("subscriber_id").orderBy("session_start", "session_id")

lag_df = timestamp_df \
    .filter(F.col("subscriber_id").isNotNull()) \
    .withColumn("previous_watch_minutes", F.lag("watch_minutes", 1).over(lag_window)) \
    .withColumn("change_in_watch_minutes", F.col("watch_minutes") - F.col("previous_watch_minutes"))

lag_df.filter(F.col("subscriber_id") == "SUB00001") \
    .select("subscriber_id", "session_id", "session_start", "watch_minutes", "previous_watch_minutes", "change_in_watch_minutes") \
    .orderBy("session_start", "session_id") \
    .show()

+-------------+----------+-------------------+-------------+----------------------+-----------------------+
|subscriber_id|session_id|      session_start|watch_minutes|previous_watch_minutes|change_in_watch_minutes|
+-------------+----------+-------------------+-------------+----------------------+-----------------------+
|     SUB00001|WS00064390|2026-01-02 17:32:00|           58|                  NULL|                   NULL|
|     SUB00001|WS00045183|2026-01-05 04:51:00|           78|                    58|                     20|
|     SUB00001|WS00028484|2026-01-09 08:46:00|            4|                    78|                    -74|
|     SUB00001|WS00043555|2026-01-24 19:41:00|          108|                     4|                    104|
|     SUB00001|WS00041862|2026-02-18 07:43:00|           79|                   108|                    -29|
|     SUB00001|WS00059616|2026-02-25 07:38:00|           18|                    79|                    -61|
|     SUB00001|WS00042345|20

In [29]:
# Task 20 - Combine batches with unionByName(); remove duplicate session_id; compare totals
combined_df = sessions_df.unionByName(extra_df)

unique_df = combined_df.dropDuplicates(["session_id"])

print("Main:", sessions_df.count())
print("Extra:", extra_df.count())
print("Combined:", combined_df.count())
print("Unique session_id values:", unique_df.count())
print("Duplicates removed:", combined_df.count() - unique_df.count())

Main: 80000
Extra: 10000
Combined: 90000
Unique session_id values: 89400
Duplicates removed: 600


In [30]:
# Task 21 - Lazy pipeline (filter, join, withColumn, groupBy, orderBy) inspected with explain()
pipeline_df = sessions_df \
    .filter(F.col("session_status") == "Completed") \
    .join(content_df, on="content_id", how="inner") \
    .withColumn("watch_hours", F.col("watch_minutes") / 60) \
    .groupBy("genre") \
    .agg(
        F.count("*").alias("completed_sessions"),
        F.round(F.sum("watch_hours"), 2).alias("total_watch_hours")
    ) \
    .orderBy(F.col("total_watch_hours").desc())

pipeline_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_watch_hours#1332 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_watch_hours#1332 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=3444]
      +- HashAggregate(keys=[genre#100], functions=[count(1), sum(watch_hours#1330)])
         +- Exchange hashpartitioning(genre#100, 200), ENSURE_REQUIREMENTS, [plan_id=3441]
            +- HashAggregate(keys=[genre#100], functions=[partial_count(1), partial_sum(watch_hours#1330)])
               +- Project [genre#100, (cast(watch_minutes#22 as double) / 60.0) AS watch_hours#1330]
                  +- BroadcastHashJoin [content_id#19], [content_id#98], Inner, BuildRight, false
                     :- Project [content_id#19, watch_minutes#22]
                     :  +- Filter ((isnotnull(session_status#26) AND (session_status#26 = Completed)) AND isnotnull(content_id#19))
                     :     +- FileScan csv [content_id#19,watch_minutes#22,session_status

In [31]:
# Task 21 - Trigger the pipeline with an action
pipeline_df.show()

+-----------+------------------+-----------------+
|      genre|completed_sessions|total_watch_hours|
+-----------+------------------+-----------------+
|  Adventure|              1263|          1909.48|
|      Drama|               890|          1218.05|
|  Animation|               918|          1097.53|
|     Comedy|               696|           979.57|
|     Sci-Fi|               739|           956.08|
|     Sports|               829|           906.38|
|Documentary|               587|           588.28|
|   Thriller|               470|           537.93|
+-----------+------------------+-----------------+



In [32]:
# Task 22 - Cache completed sessions; run count() twice and compare timings
completed_df = sessions_df.filter(F.col("session_status") == "Completed")

completed_df.cache()

print("Is cached before any action:", completed_df.is_cached)

start = time.time()
first_count = completed_df.count()
first_count_time = time.time() - start

start = time.time()
second_count = completed_df.count()
second_count_time = time.time() - start

print("First count:", first_count, "-", round(first_count_time, 4), "seconds")
print("Second count:", second_count, "-", round(second_count_time, 4), "seconds")

Is cached before any action: True
First count: 6392 - 0.9958 seconds
Second count: 6392 - 0.5155 seconds


In [33]:
# Task 22 - Run the aggregation twice on cached data and compare timings
def run_aggregation():
    return completed_df.groupBy("stream_quality") \
        .agg(
            F.count("*").alias("sessions"),
            F.round(F.avg("watch_minutes"), 2).alias("avg_watch_minutes")
        ) \
        .orderBy("stream_quality") \
        .collect()

start = time.time()
first_result = run_aggregation()
first_agg_time = time.time() - start

start = time.time()
second_result = run_aggregation()
second_agg_time = time.time() - start

print("First aggregation:", round(first_agg_time, 4), "seconds")
print("Second aggregation:", round(second_agg_time, 4), "seconds")

for row in second_result:
    print(row["stream_quality"], row["sessions"], row["avg_watch_minutes"])

First aggregation: 0.5302 seconds
Second aggregation: 0.2878 seconds
4K 599 77.31
Full HD 2120 77.53
HD 2570 76.4
SD 1103 76.69


In [34]:
# Task 22 - Release the cached completed sessions
completed_df.unpersist(blocking=True)

print("Is cached:", completed_df.is_cached)

Is cached: False


In [35]:
# Task 23 - Persist 4K sessions with MEMORY_AND_DISK; run two actions
quality_4k_df = sessions_df.filter(F.col("stream_quality") == "4K")

quality_4k_df.persist(StorageLevel.MEMORY_AND_DISK)

print("4K sessions:", quality_4k_df.count())

quality_4k_df.groupBy("session_status").count().orderBy(F.col("count").desc()).show()

print(quality_4k_df.storageLevel)

4K sessions: 7312
+--------------+-----+
|session_status|count|
+--------------+-----+
|   Interrupted| 5457|
|     Abandoned| 1256|
|     Completed|  599|
+--------------+-----+

Disk Memory Serialized 1x Replicated


In [36]:
# Task 23 - Release the persisted data using unpersist()
quality_4k_df.unpersist(blocking=True)

print("Is cached:", quality_4k_df.is_cached)

Is cached: False


In [37]:
# Task 24 - Repartition into 8 partitions; row counts per partition (spark_partition_id)
sessions_8_df = sessions_df.repartition(8)

print("Partitions:", sessions_8_df.rdd.getNumPartitions())

sessions_8_df.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

Partitions: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|           0|10000|
|           1| 9999|
|           2|10001|
|           3|10000|
|           4|10000|
|           5|10000|
|           6|10000|
|           7|10000|
+------------+-----+



In [38]:
# Task 25 - From 8 partitions: compare coalesce(2) vs repartition(2) with explain()
coalesced_2_df = sessions_8_df.coalesce(2)
repartitioned_2_df = sessions_8_df.repartition(2)

print("Coalesce(2):", coalesced_2_df.rdd.getNumPartitions())
print("Repartition(2):", repartitioned_2_df.rdd.getNumPartitions())

print("COALESCE PLAN")
coalesced_2_df.explain()

print("REPARTITION PLAN")
repartitioned_2_df.explain()

Coalesce(2): 2
Repartition(2): 2
COALESCE PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- Coalesce 2
      +- ShuffleQueryStage 0
         +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=4160]
            +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_status#26,data_used_mb#27,user_rating#28] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/watch_sessions_80000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<session_id:string,subscriber_id:string,content_id:string,device_id:string,session_start:ti...
+- == Initial Plan ==
   Coalesce 2
   +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=4160]
      +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minu

In [39]:
# Task 25 - Write chosen output DataFrame (deduplicated sessions) to Parquet
unique_df.repartition(2) \
    .write.mode("overwrite") \
    .parquet("unique_sessions_parquet")

In [40]:
# Task 25 - Inspect the resulting Parquet part files
for name in sorted(os.listdir("unique_sessions_parquet")):
    path = os.path.join("unique_sessions_parquet", name)
    print(name, "-", os.path.getsize(path), "bytes")

spark.read.parquet("unique_sessions_parquet").count()

._SUCCESS.crc - 8 bytes
.part-00000-0f3d4991-fd3e-465b-805e-833cb37b2994-c000.snappy.parquet.crc - 8768 bytes
.part-00001-0f3d4991-fd3e-465b-805e-833cb37b2994-c000.snappy.parquet.crc - 8764 bytes
_SUCCESS - 0 bytes
part-00000-0f3d4991-fd3e-465b-805e-833cb37b2994-c000.snappy.parquet - 1121144 bytes
part-00001-0f3d4991-fd3e-465b-805e-833cb37b2994-c000.snappy.parquet - 1120320 bytes


89400

**Streaming performance report**

In [41]:
# Capstone Step 01 - Union the two batches and deduplicate by session_id
combined_df = sessions_df.unionByName(extra_df)

unique_sessions_df = combined_df.dropDuplicates(["session_id"])

print("Main batch:", sessions_df.count())
print("Extra batch:", extra_df.count())
print("Combined:", combined_df.count())
print("Unique sessions:", unique_sessions_df.count())

Main batch: 80000
Extra batch: 10000
Combined: 90000
Unique sessions: 89400


In [42]:
# Capstone Step 02 - Check relevant nulls before cleaning
unique_sessions_df.select(
    F.count(F.when(F.col("subscriber_id").isNull(), 1)).alias("null_subscriber_id"),
    F.count(F.when(F.col("device_id").isNull(), 1)).alias("null_device_id"),
    F.count(F.when(F.col("completion_pct").isNull(), 1)).alias("null_completion_pct"),
    F.count(F.when(F.col("buffering_seconds").isNull(), 1)).alias("null_buffering_seconds"),
    F.count(F.when(F.col("data_used_mb").isNull(), 1)).alias("null_data_used_mb")
).show()

+------------------+--------------+-------------------+----------------------+-----------------+
|null_subscriber_id|null_device_id|null_completion_pct|null_buffering_seconds|null_data_used_mb|
+------------------+--------------+-------------------+----------------------+-----------------+
|              1061|          2237|               3928|                  4933|                0|
+------------------+--------------+-------------------+----------------------+-----------------+



In [43]:
# Capstone Step 02/03 - Null handling (region -> Unknown) and joins with subscribers and content; add month
enriched_df = unique_sessions_df \
    .join(subscribers_df.select("subscriber_id", "region"), on="subscriber_id", how="left") \
    .withColumn("region", F.coalesce(F.col("region"), F.lit("Unknown"))) \
    .join(content_df.select("content_id", "content_title", "genre"), on="content_id", how="inner") \
    .withColumn("session_start", F.to_timestamp("session_start", "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("month", F.date_format("session_start", "yyyy-MM")) \
    .withColumn("is_completed", F.when(F.col("session_status") == "Completed", 1).otherwise(0))

print("Rows after joins:", enriched_df.count())

enriched_df.groupBy("region").count().orderBy(F.col("count").desc()).show()

Rows after joins: 89400
+-------+-----+
| region|count|
+-------+-----+
|  South|24890|
|   West|21927|
|  North|18574|
|   East|12659|
|Central| 9630|
|Unknown| 1720|
+-------+-----+



In [44]:
# Capstone Step 04 - Monthly metrics by region and genre
report_df = enriched_df \
    .groupBy("month", "region", "genre") \
    .agg(
        F.count("*").alias("session_count"),
        F.round(F.sum("watch_minutes") / 60, 2).alias("watch_hours"),
        F.round(F.sum("is_completed") / F.count("*") * 100, 2).alias("completion_rate_pct"),
        F.round(F.avg("buffering_seconds"), 2).alias("avg_buffering_seconds"),
        F.round(F.sum("data_used_mb"), 2).alias("total_data_used_mb")
    )

report_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[month#3957, region#3955, genre#100], functions=[count(1), sum(watch_minutes#4154), sum(is_completed#3958), avg(buffering_seconds#4158), sum(data_used_mb#4164)])
   +- Exchange hashpartitioning(month#3957, region#3955, genre#100, 200), ENSURE_REQUIREMENTS, [plan_id=5407]
      +- HashAggregate(keys=[month#3957, region#3955, genre#100], functions=[partial_count(1), partial_sum(watch_minutes#4154), partial_sum(is_completed#3958), partial_avg(buffering_seconds#4158), partial_sum(data_used_mb#4164)])
         +- Project [watch_minutes#4154, buffering_seconds#4158, data_used_mb#4164, region#3955, genre#100, date_format(gettimestamp(session_start#4152, yyyy-MM-dd HH:mm:ss, TimestampType, try_to_timestamp, Some(Etc/UTC), true), yyyy-MM, Some(Etc/UTC)) AS month#3957, CASE WHEN (session_status#4162 = Completed) THEN 1 ELSE 0 END AS is_completed#3958]
            +- BroadcastHashJoin [content_id#4148], [content_id#98],

In [45]:
# Capstone Step 05 - Top 3 content titles by watch hours within each genre
title_hours_df = enriched_df \
    .groupBy("genre", "content_title") \
    .agg(F.round(F.sum("watch_minutes") / 60, 2).alias("watch_hours"))

title_window = Window.partitionBy("genre").orderBy(F.col("watch_hours").desc())

top_titles_df = title_hours_df \
    .withColumn("title_rank", F.dense_rank().over(title_window)) \
    .filter(F.col("title_rank") <= 3)

top_titles_df.orderBy("genre", "title_rank").show()

+-----------+----------------+-----------+----------+
|      genre|   content_title|watch_hours|title_rank|
+-----------+----------------+-----------+----------+
|  Adventure|Infinite Horizon|     1210.8|         1|
|  Adventure|    Solar Memory|    1129.47|         2|
|  Adventure|    Lost Journey|    1127.85|         3|
|  Animation|      Last River|      969.3|         1|
|  Animation| Emerald Circuit|     963.08|         2|
|  Animation|   Golden Puzzle|     932.22|         3|
|     Comedy|   Bright Valley|     1173.8|         1|
|     Comedy|    Blue Horizon|    1151.25|         2|
|     Comedy|     Wild Canvas|    1136.23|         3|
|Documentary|   Fading Signal|     878.68|         1|
|Documentary|  Emerald Valley|     743.88|         2|
|Documentary|   Solar Horizon|     684.27|         3|
|      Drama|  Crimson Summit|     1225.0|         1|
|      Drama|  Rising Archive|    1067.07|         2|
|      Drama|     Wild Harbor|     855.93|         3|
|     Sci-Fi|     Lost Canva

In [46]:
# Capstone Step 06 - Cache the final report and run the first action
report_df.cache()

print("Report rows:", report_df.count())

report_df.filter((F.col("month") == "2026-01") & (F.col("region") == "Central")) \
    .orderBy("genre") \
    .show(truncate=False)

print("Is cached:", report_df.is_cached)

Report rows: 432
+-------+-------+-----------+-------------+-----------+-------------------+---------------------+------------------+
|month  |region |genre      |session_count|watch_hours|completion_rate_pct|avg_buffering_seconds|total_data_used_mb|
+-------+-------+-----------+-------------+-----------+-------------------+---------------------+------------------+
|2026-01|Central|Adventure  |203          |182.1      |7.39               |35.41                |188882.1          |
|2026-01|Central|Animation  |173          |107.35     |8.09               |32.47                |120522.6          |
|2026-01|Central|Comedy     |122          |103.27     |9.02               |34.5                 |110126.4          |
|2026-01|Central|Documentary|120          |59.8       |0.83               |27.1                 |76352.2           |
|2026-01|Central|Drama      |126          |106.45     |9.52               |40.63                |126110.6          |
|2026-01|Central|Sci-Fi     |132          |99.3

In [47]:
# Capstone Step 06 - Second action on the cached report
report_df.groupBy("region") \
    .agg(
        F.sum("session_count").alias("session_count"),
        F.round(F.sum("watch_hours"), 2).alias("watch_hours"),
        F.round(F.sum("total_data_used_mb"), 2).alias("total_data_used_mb")
    ) \
    .orderBy(F.col("watch_hours").desc()) \
    .show()

+-------+-------------+-----------+------------------+
| region|session_count|watch_hours|total_data_used_mb|
+-------+-------------+-----------+------------------+
|  South|        24890|   18816.67|      2.09717845E7|
|   West|        21927|   16684.67|      1.84299963E7|
|  North|        18574|   13848.42|      1.53808358E7|
|   East|        12659|    9624.54|      1.06340338E7|
|Central|         9630|    7278.18|         8152306.9|
|Unknown|         1720|    1278.93|         1407575.3|
+-------+-------------+-----------+------------------+



In [48]:
# Capstone Step 06 - Release the cached report (unpersist)
report_df.unpersist(blocking=True)

print("Is cached:", report_df.is_cached)

Is cached: False


In [49]:
# Capstone Step 07 - Save the final report as Parquet with 3 output partitions
report_df.repartition(3) \
    .write.mode("overwrite") \
    .parquet("streaming_performance_report")

In [50]:
# Capstone Step 07 - Verify Parquet output (part files, row count, preview)
for name in sorted(os.listdir("streaming_performance_report")):
    path = os.path.join("streaming_performance_report", name)
    print(name, "-", os.path.getsize(path), "bytes")

saved_df = spark.read.parquet("streaming_performance_report")

print("Rows read back:", saved_df.count())

saved_df.orderBy("month", "region", "genre").show()

._SUCCESS.crc - 8 bytes
.part-00000-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet.crc - 64 bytes
.part-00001-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet.crc - 64 bytes
.part-00002-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet.crc - 64 bytes
_SUCCESS - 0 bytes
part-00000-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet - 6723 bytes
part-00001-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet - 6817 bytes
part-00002-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet - 6699 bytes
Rows read back: 432
+-------+-------+-----------+-------------+-----------+-------------------+---------------------+------------------+
|  month| region|      genre|session_count|watch_hours|completion_rate_pct|avg_buffering_seconds|total_data_used_mb|
+-------+-------+-----------+-------------+-----------+-------------------+---------------------+------------------+
|2026-01|Central|  Adventure|          203|      182.1|               7.39|                

In [51]:
# Submission - Zip the Parquet output directory
import shutil

shutil.make_archive("streaming_performance_report", "zip", "streaming_performance_report")

'/content/streaming_performance_report.zip'

In [52]:
# Submission - List the contents of the zip
import zipfile

with zipfile.ZipFile("streaming_performance_report.zip") as z:
    for name in z.namelist():
        print(name)

.part-00000-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet.crc
.part-00001-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet.crc
_SUCCESS
.part-00002-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet.crc
part-00002-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet
part-00001-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet
._SUCCESS.crc
part-00000-62fba617-3a1d-47a9-96ab-4fd9d3eba6a5-c000.snappy.parquet


In [53]:
# Submission - Download the zip
from google.colab import files

files.download("streaming_performance_report.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>